# Imports

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, classification_report
)
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

# Config

In [6]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "microsoft/deberta-v3-large"
CSV_PATH = "../../datasets/processed/final_combined.csv"

# Load Dataset

In [ ]:
df = pd.read_csv("../../datasets/processed/final_combined.csv")
headers_df = df[["headers_raw","label"]].copy()

# Remove empty/missing
headers_df["headers_raw"] = headers_df["headers_raw"].fillna("").astype(str)
headers_df = headers_df[headers_df["headers_raw"].str.strip().astype(bool)]

# Drop duplicates
headers_df = headers_df.drop_duplicates(subset=["headers_raw"])

print("Header-only rows:", headers_df.shape[0])
print(headers_df["label"].value_counts(normalize=True))

train_df, val_df = train_test_split(
    headers_df,
    test_size=0.1,
    random_state=SEED,
    stratify=headers_df["label"]
)

print("Train size:", len(train_df), "Val size:", len(val_df))
print("Train label counts:", train_df["label"].value_counts())
print("Val label counts:", val_df["label"].value_counts())

Train: 74643, Val: 8294
Train label counts: label
1    47343
0    27300
Name: count, dtype: int64
Val label counts: label
1    5260
0    3034
Name: count, dtype: int64


/tmp/ipykernel_184903/2382797235.py:1: DtypeWarning: Columns (3,4,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


# Tokenizer and DataCollator

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Assigned pad_token =", tokenizer.pad_token)

def tokenize_headers(batch):
    return tokenizer(
        batch["headers_raw"],
        truncation=True,
        max_length=512,             # if you change here, you should also change in the inference function
        padding="max_length",
    )

from datasets import Dataset
train_ds = Dataset.from_pandas(train_df.rename(columns={"label":"labels"}))
val_ds   = Dataset.from_pandas(val_df.rename(columns={"label":"labels"}))
train_ds_tokenized = train_ds.map(tokenize_headers, batched=True, remove_columns=UNIFIED_COLUMNS)
val_ds_tokenized   = val_ds.map(tokenize_headers, batched=True, remove_columns=UNIFIED_COLUMNS)
data_collator = DataCollatorWithPadding(tokenizer)

/home/defalt/.pyenv/versions/venv-3.10.12/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


ValueError: Column to remove ['label'] not in the dataset. Current columns in the dataset: ['id', 'subject', 'body_text', 'attachment_text', 'headers_raw', 'from_email', 'to_email', 'reply_to_email', 'date', 'urls', 'header_from_reply_mismatch', 'header_domain_mismatch', 'header_suspicious_tld', 'header_received_count', 'header_to_anomaly', 'header_x_mailer_anomaly', 'header_date_malformed', 'labels', 'source', '__index_level_0__']

# Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.config.pad_token_id = tokenizer.pad_token_id
model.gradient_checkpointing_enable()  # Gradient checkpointing
model.to(DEVICE)

# Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# Trainning Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="header_deberta_large_out",

    evaluation_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-5,                # safer than 5e-5
    lr_scheduler_type="linear",
    warmup_ratio=0.06,                 # 6% warmup (standard for large models)

    per_device_train_batch_size=2,     # safe for v3-large + gradient checkpointing
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,     # effective batch = 16

    num_train_epochs=3,
    weight_decay=0.01,

    bf16=True,                         # best for DeBERTa
    fp16=False,                        # avoid fp16 instability

    max_grad_norm=1.0,
    gradient_checkpointing=True,       # required for v3-large

    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",

    optim="adamw_torch",
    label_smoothing_factor=0.1,        # improves robustness
    report_to="none",
)


# Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_tokenized,
    eval_dataset=val_ds_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train

In [ ]:
trainer.train()

# Save Model

In [ ]:
trainer.save_model("header_deberta_model")
tokenizer.save_pretrained("header_deberta_tokenizer")
print("Stage-4 DeBERTa header model saved.")

# Evaluate

In [ ]:
preds = trainer.predict(val_ds_tokenized)
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)
probs  = torch.softmax(torch.tensor(preds.predictions), dim=1)[:, 1].numpy()

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legit","Phishing"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix - Header DeBERTa Model")
plt.show()

# Loss Curve
logs = trainer.state.log_history
df_logs = pd.DataFrame(logs)
loss_df = df_logs[df_logs["loss"].notna()]
plt.figure(figsize=(10,5))
plt.plot(loss_df["step"], loss_df["loss"], marker="o")
plt.title("Training Loss Curve")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

# Metrics Curve (Accuracy / Precision / Recall / F1 per epoch)
eval_df = df_logs[df_logs["eval_loss"].notna()].copy()
if "epoch" not in eval_df.columns:
    eval_df["epoch"] = range(1, len(eval_df)+1)

plt.figure(figsize=(14,6))
for col, label in [
    ("eval_accuracy","Accuracy"),
    ("eval_precision","Precision"),
    ("eval_recall","Recall"),
    ("eval_f1","F1 Score"),
]:
    if col in eval_df:
        plt.plot(eval_df["epoch"], eval_df[col], marker="o", label=label)
plt.xlabel("Epoch")
plt.ylabel("Metric")
plt.title("Evaluation Metrics per Epoch - Header DeBERTa Model")
plt.legend()
plt.grid(True)
plt.show()

# Precision-Recall Curve
prec, rec, thresh = precision_recall_curve(y_true, probs)
plt.figure(figsize=(8,4))
plt.plot(rec, prec)
plt.title("Precision-Recall Curve - Header DeBERTa Model")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.grid(True)
plt.show()

# Classification Report
print(classification_report(y_true, y_pred, target_names=["Legit","Phishing"]))

# Inference Helper

In [ ]:
def predict_header_eml(eml_path: str):
    loader = EmlLoader()
    _, _, headers_raw = loader.separate_header_blocks(eml_path)
    if not headers_raw:
        return {"error": "Cannot read headers from EML."}

    model.eval()
    inputs = tokenizer(
        [headers_raw],
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()
        pred   = int(np.argmax(probs))

    return {
        "prediction": "PHISHING" if pred==1 else "LEGIT",
        "prob_legit": float(probs[0]),
        "prob_phishing": float(probs[1]),
    }

# Example usage:
# predict_header_eml("datasets/eml_samples/sample.eml")